# PyTorch tutorial

We’ll use the FashionMNIST dataset to train a neural network that predicts if an input image belongs to one of the following classes: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, or Ankle boot.

In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

- Download training data

In [7]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [8]:
batch_size = 64  # batch is groub of

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


- Creating Models

In [9]:
# Select the device to run the model on (GPU if available, otherwise CPU)
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define a Neural Network model by inheriting from nn.Module
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()  # Initialize the parent nn.Module

        # Flatten layer converts image tensors from [N, 1, 28, 28] to [N, 784]
        self.flatten = nn.Flatten()

        # Sequential container for fully connected layers and activation functions
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),  # First fully connected layer
            nn.ReLU(),               # ReLU activation function
            nn.Linear(512, 512),      # Second fully connected layer
            nn.ReLU(),               # ReLU activation function
            nn.Linear(512, 10)        # Output layer (10 class logits)
        )

    def forward(self, x):
        # Forward pass of the network
        x = self.flatten(x)                 # Flatten the input image
        logits = self.linear_relu_stack(x)  # Pass data through the network
        return logits                       # Return raw output scores (logits)

# Create an instance of the model and move it to the selected device
model = NeuralNetwork().to(device)

# Print the model architecture
print(model)


Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


- Optimizing the Model Parameters

In [10]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [11]:
def train(dataloader, model, loss_fn, optimizer):
    # Get the total number of samples in the training dataset
    size = len(dataloader.dataset)

    # Set the model to training mode
    # This enables behaviors like dropout and batch normalization (if used)
    model.train()

    # Iterate over the DataLoader, which returns batches of (inputs, labels)
    for batch, (X, y) in enumerate(dataloader):
        # Move input data and labels to the same device as the model (CPU/GPU)
        X, y = X.to(device), y.to(device)

        # =========================
        # Forward pass
        # =========================

        # Compute model predictions (logits)
        pred = model(X)

        # Compute the loss between predictions and true labels
        loss = loss_fn(pred, y)

        # =========================
        # Backpropagation
        # =========================

        # Compute gradients of the loss with respect to model parameters
        loss.backward()

        # Update model parameters using the optimizer
        optimizer.step()

        # Reset gradients to zero before the next iteration
        optimizer.zero_grad()

        # =========================
        # Logging
        # =========================

        # Print loss every 100 batches
        if batch % 100 == 0:
            loss_value = loss.item()                # Convert loss tensor to Python number
            current = (batch + 1) * len(X)          # Number of samples processed so far
            print(f"loss: {loss_value:>7f}  [{current:>5d}/{size:>5d}]")


In [12]:
def test(dataloader, model, loss_fn):
    # Get the total number of samples in the test dataset
    size = len(dataloader.dataset)

    # Get the number of batches in the DataLoader
    num_batches = len(dataloader)

    # Set the model to evaluation mode
    # This disables behaviors like dropout and batch normalization updates
    model.eval()

    # Initialize variables to accumulate loss and correct predictions
    test_loss, correct = 0, 0

    # Disable gradient computation to save memory and computation
    with torch.no_grad():
        # Iterate over the test DataLoader
        for X, y in dataloader:
            # Move input data and labels to the same device as the model
            X, y = X.to(device), y.to(device)

            # Forward pass: compute predictions (logits)
            pred = model(X)

            # Accumulate the loss for this batch
            test_loss += loss_fn(pred, y).item()

            # Count correct predictions
            # pred.argmax(1) gives the predicted class index
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    # Compute the average loss over all batches
    test_loss /= num_batches

    # Compute overall accuracy
    correct /= size

    # Print test accuracy and average loss
    print(
        f"Test Error: \n"
        f" Accuracy: {(100 * correct):>0.1f}%, Avg loss: {test_loss:>8f} \n"
    )


In [13]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.306346  [   64/60000]
loss: 2.295954  [ 6464/60000]
loss: 2.280115  [12864/60000]
loss: 2.271601  [19264/60000]
loss: 2.259527  [25664/60000]
loss: 2.228003  [32064/60000]
loss: 2.239573  [38464/60000]
loss: 2.198057  [44864/60000]
loss: 2.204009  [51264/60000]
loss: 2.178834  [57664/60000]
Test Error: 
 Accuracy: 45.4%, Avg loss: 2.170625 

Epoch 2
-------------------------------
loss: 2.178730  [   64/60000]
loss: 2.173702  [ 6464/60000]
loss: 2.123492  [12864/60000]
loss: 2.133554  [19264/60000]
loss: 2.096565  [25664/60000]
loss: 2.032921  [32064/60000]
loss: 2.067116  [38464/60000]
loss: 1.984687  [44864/60000]
loss: 1.999091  [51264/60000]
loss: 1.936799  [57664/60000]
Test Error: 
 Accuracy: 58.6%, Avg loss: 1.932455 

Epoch 3
-------------------------------
loss: 1.961433  [   64/60000]
loss: 1.937053  [ 6464/60000]
loss: 1.831923  [12864/60000]
loss: 1.859624  [19264/60000]
loss: 1.763867  [25664/60000]
loss: 1.710085  [32064/600